# 20.7 題目和答案都在輸入裡，模型究竟練哪一段？


一道訓練題寫著「桌上有幾隻貓？」和答案「兩隻」。題目提供線索，回答才是要模仿的輸出。所以對話的監督式微調（SFT，[7.1](https://birdhackor.github.io/tiny-perceptron-vlm/7.1.html)）讓問題、圖片與前文都留在輸入裡給模型讀，只把助理的答案和答完後的結束標記算進代價；不是把題目從模型的視野裡遮掉。

下圖用手寫的 Q、A 小例子畫出這件事：上排是模型讀的輸入 X，下排是每個位置要猜的下一項目標 Y，藍色是計入代價的目標。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/natural-v4-answer-mask.svg")))

在「助理」這個位置預測 A 時，模型能讀的只有開頭到這個位置為止的前文，右邊的 A 還看不到；到了 A 的位置，才以它為前文去預測結束。這是 [7.4](https://birdhackor.github.io/tiny-perceptron-vlm/7.4.html) 的下一位置對齊：要猜的答案不能先交給負責猜它的位置。圖中有兩個「結束」：上排第 4 個位置是使用者那段話的 EOS（結束標記），只當前文；下排最後一個位置是答案後的 EOS，這個才是要學的目標。兩者是同一個 token，差別在它出現在輸入裡，還是被當成目標。

下面用 [7.1](https://birdhackor.github.io/tiny-perceptron-vlm/7.1.html) 節錄過的 `render_chat` 排出這個小例子，印出 X 與 Y：


In [ ]:
from tiny_perceptron.data import render_chat

x, y = render_chat(
    [
        {"role": "user", "content": "Q"},
        {"role": "assistant", "content": "A"},
    ]
)
active = y != -100
first = active.nonzero()[0].item()
print("輸入X", x.tolist())
print("目標Y", y.tolist())
print("有效目標數", active.sum().item())
print("第一個有效目標的位置", first)
print("有效目標ID", y[active].tolist())

這個小例子用 byte 編碼，Q 和 A 各只占一個位置，輸入共六個位置：X 是 `[1, 3, 89, 2, 4, 73]`，依序是 BOS、使用者、Q、使用者那段的 EOS（ID 2）、助理、A。Y 的前四項是 −100，也就是 7.1 的 `IGNORE`，表示這個位置不計代價（它只出現在目標裡，不是輸入的 token ID）；後兩項是 73 與 2，也就是 A 和答案後的 EOS。`y != -100` 對每個位置問「這裡計不計代價」，得到由真、假組成的 `active`；`active.nonzero()` 列出所有為真的位置，`[0]` 取第一個，`.item()` 換成普通整數。所以有效目標有 2 個，第一個在 0 起算的位置 4，正好是 X 讀到「助理」的位置。這個小例子只示範下一位置對齊，不表示完整的 Qwen 圖片輸入也只有六個位置。

練習把程式裡的 `"content": "A"` 改成 `"content": "AB"`。先預測：輸入變成七個位置，有效目標變成 3 個（A、B 與 EOS），第一個有效目標的位置仍是 4，問題 Q 仍留在輸入裡；再執行核對。

真正的資料先交給 Qwen 附的前處理器（processor）：它把圖片換成模型能讀的輸入，並套用 Qwen 的對話模板標出角色。長度要在這一步完成之後才算得準。本課訓練的序列上限是 2,048 個 token；超過上限而截掉答案的尾端或結束標記，就改變了原示範要教的工作（[7.10](https://birdhackor.github.io/tiny-perceptron-vlm/7.10.html)）。逐筆核對有效目標與保留的範圍，比只問「句子夠不夠短」更準確。
